In [1]:
%load_ext autoreload
%autoreload 2
%env TRANSFORMERS_CACHE=/data2/hluo/.cache/huggingface/hub
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

env: TRANSFORMERS_CACHE=/data2/hluo/.cache/huggingface/hub


/data2/hluo/anaconda3/envs/xllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load model & tokenizer

In [2]:
model_name = 'EleutherAI/gpt-j-6b'
# model_name = 'meta-llama/Llama-2-7b-hf'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name, device='cuda:0')
EDIT_LAYER = 9

Loading:  EleutherAI/gpt-j-6b


## Load dataset and Compute task-conditioned mean activations

In [ ]:
dataset = load_dataset('country-capital', seed=0)

In [ ]:
from baukit import TraceDict
HEADS = [f"model.layers.{i}.self_attn.attn_out" for i in range(model.config.num_hidden_layers)]
MLPS = [f"model.layers.{i}.mlp" for i in range(model.config.num_hidden_layers)]
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, tokenizer=tokenizer, query_target_pair=test_pair, prepend_bos_token=True)
prompt = create_prompt(prompt_data)
prompt = tokenizer(prompt, return_tensors='pt').input_ids.to('cuda:0')

with TraceDict(model, HEADS+MLPS) as ret:
    output = model(prompt, output_hidden_states = True)

hidden_states = output.hidden_states
hidden_states = torch.stack(hidden_states, dim = 0).squeeze()
hidden_states = hidden_states.detach().cpu().numpy()
head_wise_hidden_states = [ret[head].output.squeeze().detach().cpu() for head in HEADS]
head_wise_hidden_states = torch.stack(head_wise_hidden_states, dim = 0).squeeze().numpy()
mlp_wise_hidden_states = [ret[mlp].output.squeeze().detach().cpu() for mlp in MLPS]
mlp_wise_hidden_states = torch.stack(mlp_wise_hidden_states, dim = 0).squeeze().numpy()
head_wise_hidden_states.shape, mlp_wise_hidden_states.shape

((32, 78, 4096), (32, 78, 4096))

In [ ]:
head_wise_hidden_states.shape

(32, 32, 78, 128)

In [ ]:
mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)

100%|██████████| 100/100 [00:30<00:00,  3.26it/s]


In [ ]:
mean_activations.shape

torch.Size([32, 32, 97, 128])

## Compute function vector (FV)

In [ ]:
FV, top_heads = compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10)

## Prompt Creation - ICL, Shuffled-Label, Zero-Shot, and Natural Text

In [ ]:
# Sample ICL example pairs, and a test word
# dataset = load_dataset('country-capital')
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, tokenizer=tokenizer, query_target_pair=test_pair, prepend_bos_token=True)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), '\n\n')

shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, tokenizer=tokenizer, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, tokenizer=tokenizer, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))

ICL prompt:
 '<s>Q: Moldova\nA: Chisinau\n\nQ: Bhutan\nA: Thimphu\n\nQ: Sao Tome and Principe\nA: Sao Tome\n\nQ: Kosovo\nA: Pristina\n\nQ: Eritrea\nA: Asmara\n\nQ: Australia\nA:' 


Shuffled ICL Prompt:
 '<s>Q: Moldova\nA: Asmara\n\nQ: Bhutan\nA: Sao Tome\n\nQ: Sao Tome and Principe\nA: Pristina\n\nQ: Kosovo\nA: Chisinau\n\nQ: Eritrea\nA: Thimphu\n\nQ: Australia\nA:' 


Zero-Shot Prompt:
 '<s>Q: Australia\nA:'


## Evaluation

### Clean ICL Prompt

In [ ]:
# Check model's ICL answer
clean_logits = sentence_eval(sentence, [test_pair['output']], model, tokenizer, compute_nll=False)
print(clean_logits.shape)
print("Input Sentence:", repr(sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("ICL Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')

torch.Size([1, 32000])
Input Sentence: '<s>Q: Moldova\nA: Chisinau\n\nQ: Bhutan\nA: Thimphu\n\nQ: Sao Tome and Principe\nA: Sao Tome\n\nQ: Kosovo\nA: Pristina\n\nQ: Eritrea\nA: Asmara\n\nQ: Australia\nA:' 

Input Query: 'Australia', Target: 'Canberra'

ICL Prompt Top K Vocab Probs:
 [('Can', 0.76276), ('Sydney', 0.11172), ('Melbourne', 0.0187), ('Per', 0.01341), ('C', 0.00881)] 



### Corrupted ICL Prompt

In [ ]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<s>Q: Moldova\nA: Asmara\n\nQ: Bhutan\nA: Sao Tome\n\nQ: Sao Tome and Principe\nA: Pristina\n\nQ: Kosovo\nA: Chisinau\n\nQ: Eritrea\nA: Thimphu\n\nQ: Australia\nA:' 

Input Query: 'Australia', Target: 'Canberra'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [('T', 0.03672), ('Br', 0.02664), ('B', 0.02357), ('K', 0.02255), ('Bh', 0.02044)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [('Can', 0.38426), ('Sydney', 0.02633), ('Su', 0.02628), ('Ad', 0.02473), ('Well', 0.02295)]


### Zero-Shot Prompt

In [ ]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<s>Q: Australia\nA:' 

Input Query: 'Australia', Target: 'Canberra'

Zero-Shot Top K Vocab Probs:
 [('', 0.04282), ('I', 0.03861), ('The', 0.03258), ('Sydney', 0.03196), ('Australia', 0.02681)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [('Sydney', 0.17272), ('Melbourne', 0.07105), ('Can', 0.06044), ('Br', 0.05593), ('', 0.03564)]


### Natural Text Prompt

In [ ]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)


print("Input Sentence: ", repr(sentence))
print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "Australia" means'
GPT-J: '<s>The word "Australia" means "Southern Land" in Latin. The'
GPT-J+FV: '<s>The word "Australia" means "Southern Cross" in the flag of' 

